# Ways to Access Data Through the Denodo Data Catalog REST API


 **Methods that we can access data based on the Denodo Data Catalog Swagger UI page** (`/denodo-data-catalog/swagger-ui/index.html`).


1. **Interactive access** — using Swagger UI's "Try it out" directly
2. **Programmatic access** — the 4 authentication modes the REST API supports
3. **Endpoint families** — what kind of data each group of endpoints returns (browse, view-details, properties, query execution, permissions, export)
4. **Indirect access** — how `connectionUris` in view-details point us to JDBC / ODBC / REST / OData / GraphQL access into Virtual DataPort itself
5. **Summary + mental model** for the stakeholder deck

>  **Environment note (dev vs. production):** We have two Data Catalog hosts —
> `datacatalog-d.lanl.gov` (dev, ~15,014 views) and `datacatalog.lanl.gov` (production, ~4,418 views).
> Everything in this notebook works the same on both; only the `BASE` host changes.


---


## 0. Setup & Configuration

One config cell controls everything below. Switch `ENV` between `"dev"` and `"prod"` to point at the right catalog.

The Data Catalog webapp lives at the context path `/denodo-data-catalog`, and its Swagger documentation lives at
`https://<host>/denodo-data-catalog/swagger-ui/index.html` — the exact page this notebook is based on.

In [8]:
import requests
import json
import base64
from getpass import getpass


# Environment selection — THE single most important config line.
# dev  -> datacatalog-d.lanl.gov  (~15,014 views in past snapshots)
# prod -> datacatalog.lanl.gov    (~4,418 views in past snapshots)

ENV = "dev"   # "dev" or "prod"

HOSTS = {
    "dev":  "https://datacatalog-d.lanl.gov",
    "prod": "https://datacatalog.lanl.gov",
}

BASE = HOSTS[ENV] + "/denodo-data-catalog"

# Most endpoints require the internal ID of the VDP server registered
# in the Data Catalog. Section 2.5 shows how to discover this value.
SERVER_ID = 1


# set to False ONLY for quick dev testing (never in production code).
VERIFY_TLS = True

print(f"Environment : {ENV}")
print(f"Base URL    : {BASE}")
print(f"Swagger UI  : {HOSTS[ENV]}/denodo-data-catalog/swagger-ui/index.html")

Environment : dev
Base URL    : https://datacatalog-d.lanl.gov/denodo-data-catalog
Swagger UI  : https://datacatalog-d.lanl.gov/denodo-data-catalog/swagger-ui/index.html


---
## 1. Method 1 — Interactive Access via Swagger UI Itself

**What it is.** The Swagger UI page is not just documentation — it is a *live REST client*. Every endpoint listed has a **"Try it out"** button that:

1. Opens an editable form for the endpoint's path/query/body parameters,
2. Executes a **real HTTP request** against the catalog using your **current browser session** (the JSESSIONID cookie + anti-CSRF token you got when you logged into the Data Catalog web UI),
3. Shows you the exact `curl` command, the request URL, response headers, and the JSON body.

**Why it matters.**

| Strength | Limitation |
|---|---|
| Zero setup — auth is inherited from our browser login | Not automatable; session expires |
| Shows the *exact* request/response shape (great for designing `denodo.py`) | Manual, one call at a time |
| The generated `curl` command is copy-pasteable into scripts | Large responses can freeze the browser tab |
| Safe way to explore unfamiliar endpoints (e.g., property-management) | Results live only in the browser |

**Practical workflow (the one you've been using):**
1. Log into the Data Catalog web UI first (establishes the session).
2. Open `/denodo-data-catalog/swagger-ui/index.html` in the *same browser*.
3. Expand an endpoint group → "Try it out" → fill parameters → **Execute**.
4. Copy the `curl` line into your notes; later translate it into `requests` calls (Sections 2–3).

> **Tip:** the `curl` command Swagger generates includes the live cookie and XSRF token — that's exactly the pair of values the *stateful* programmatic flows in Section 2 reproduce in code.



---
## 2. Method 2 — Programmatic Access (4 Authentication Modes)

The Data Catalog REST API supports **stateless** methods (credentials sent on *every* request) and **stateful** methods (credentials sent *once*, then a session cookie + anti-CSRF token carry the session).

| # | Mode | State | Credential | Enabled by default? | Best for |
|---|------|-------|-----------|--------------------|----------|
| 2.1 | HTTP Basic | Stateless | username:password (Base64) |  Yes | Quick scripts, admin tooling |
| 2.2 | OAuth 2.0 stateless | Stateless | Bearer token every call |  No — needs `oauth.statelessEnabled=true` | Service-to-service automation |
| 2.3 | OAuth 2.0 stateful | Stateful | Bearer token once → session cookie |  No — needs `oauth.statefulEnabled=true` | ** `Authorization_code_flow` notebook** |
| 2.4 | Local login | Stateful | username/password once → session cookie |  Yes | Fallback / legacy |

Both OAuth switches live in `<DENODO_HOME>/conf/data-catalog/DataCatalogBackend.properties` — **server-side config controlled by Maxen's team**, not something we can change client-side. If a mode returns `401/403`, the most likely cause is that its switch is off, not that our token is bad.

### 2.1 HTTP Basic Authentication (stateless)

**How it works** (per RFC 7617, as implemented by Denodo):

1. Build the *user-pass* string: `username` + `:` + `password`
2. Base64-encode it into US-ASCII
3. Send header `Authorization: Basic <BASE64-user-pass>` **on every request**

**Reference example from Denodo docs** (retrieving the catalog home page):

```bash
curl --location --request GET 'http://localhost:9090/denodo-data-catalog/public/api/home' \
     --header 'Authorization: Basic YWRtaW46YWRtaW4='
```

(`YWRtaW46YWRtaW4=` is just `admin:admin` Base64-encoded.)



In [2]:
# 2.1 HTTP Basic (stateless)
# requests can build the header for us via the `auth` parameter,
# but the manual construction is shown too so we can see what's
# actually on the wire.

def basic_auth_header(username: str, password: str) -> dict:
    """Build the Authorization: Basic header manually (RFC 7617)."""
    user_pass = f"{username}:{password}"
    b64 = base64.b64encode(user_pass.encode("ascii")).decode("ascii")
    return {"Authorization": f"Basic {b64}"}

def try_basic_auth():
    username = input("Denodo username: ")
    password = getpass("Denodo password: ")

    url = f"{BASE}/public/api/home"
    resp = requests.get(
        url,
        headers={**basic_auth_header(username, password),
                 "Accept": "application/json"},
        verify=VERIFY_TLS,
    )
    print("Status:", resp.status_code)
    if resp.ok:
        print(json.dumps(resp.json(), indent=2)[:1500])
    elif resp.status_code in (401, 403):
        print("-> Basic auth appears to be DISABLED (or wrong creds). "
              "Use the OAuth flows in 2.2 / 2.3 instead.")
    return resp

# Uncomment to run interactively:
# resp = try_basic_auth()

### 2.2 OAuth 2.0 — Stateless Mode

**How it works:** attach the OAuth access token as a Bearer header **on every single request**. No session, no cookies, no CSRF token - each call is fully self-contained.

**Reference example from Denodo docs:**

```bash
curl --location --request GET \
  'http://localhost:8080/denodo-data-catalog/public/api/browse/databases?databaseName=admin&serverId=1' \
  --header 'Authorization: Bearer oauth_token' \
  --header 'Accept: application/json'
```

**Key facts:**
-  **Disabled by default.** The server admin must set `oauth.statelessEnabled=true` in `DataCatalogBackend.properties`.
- OAuth must also be enabled on the Virtual DataPort side.
- This is the mode Denodo's own **AI SDK** effectively relies on for its calls to `/public/api/askaquestion/*` (see Section 3.4).
- **Simplest automation model** — a cron job or backend service just needs a fresh token; no cookie bookkeeping.


In [3]:
#  2.2 OAuth 2.0 stateless 
# Assumes we already obtained an access token via the
# authorization-code flow (see your Authorization_code_flow notebook).

OAUTH_TOKEN = "<paste-access-token-here>"   

def oauth_stateless_get(path: str, params: dict | None = None):
    """GET any catalog endpoint with a Bearer token on this single call."""
    url = f"{BASE}{path}"
    resp = requests.get(
        url,
        params=params or {},
        headers={
            "Authorization": f"Bearer {OAUTH_TOKEN}",
            "Accept": "application/json",
        },
        verify=VERIFY_TLS,
    )
    print(resp.status_code, resp.reason, "-", resp.url)
    if resp.status_code in (401, 403):
        print("-> Either the token is expired/invalid, OR the server has "
              "oauth.statelessEnabled=false (default). Ask Maxen which.")
    return resp

# Example — browse databases on the registered VDP server:
# resp = oauth_stateless_get(
#     "/public/api/browse/databases",
#     params={"databaseName": "admin", "serverId": SERVER_ID},
# )
# resp.json()

### 2.3 OAuth 2.0 — Stateful Mode (the 3-step dance your notebook does)

This is the flow implemented in `Authorization_code_flow_copy_1.ipynb` (for develop) /  fixed `Authorization_code_flow_fixed.ipynb` (for production). We trade the OAuth token **once** for a server-side session, then ride the session.

**Step 1 — Bootstrap the session (no auth needed).**
Call any unauthenticated endpoint, e.g. `/login/configuration` (it also usefully returns the list of available VDP servers). The response sets:
- a `JSESSIONID` cookie (session identifier)
- an `X-XSRF-TOKEN` header (anti-CSRF token)

```bash
curl --location --request GET 'http://localhost:8080/denodo-data-catalog/login/configuration' \
     --header 'Accept: application/json'
# Response: Set-Cookie: JSESSIONID=1234   X-XSRF-TOKEN: abcd
```

**Step 2 — Exchange the OAuth token for an authenticated session.**
Call `/accesstoken?serverId=N` sending **all three**: the Bearer token, the Step-1 cookie, and the Step-1 XSRF token. The response rotates both values — you must capture the **new** JSESSIONID and **new** X-XSRF-TOKEN.

```bash
curl --location --request GET 'http://localhost:8080/denodo-data-catalog/accesstoken?serverId=1' \
     --header 'Accept: application/json' \
     --header 'Authorization: Bearer oauth_token' \
     --header 'Cookie: JSESSIONID=1234' \
     --header 'X-XSRF-TOKEN: abcd'
# Response: Set-Cookie: JSESSIONID=5678   X-XSRF-TOKEN: efgh
```

**Step 3 — Call the API with the session pair.**
Every subsequent request carries `Cookie: JSESSIONID=<new>` and `X-XSRF-TOKEN: <new>` — **no Bearer token needed anymore**.

```bash
curl --location --request GET \
  'http://localhost:8080/denodo-data-catalog/public/api/browse/databases?databaseName=admin&serverId=1' \
  --header 'Cookie: JSESSIONID=5678' --header 'X-XSRF-TOKEN: efgh' \
  --header 'Accept: application/json'
```

**Key facts:**
-  Disabled by default — needs `oauth.statefulEnabled=true` server-side. (Since your notebook works against the LANL dev catalog, this switch is evidently **on** there.)
- But `requests.Session()` handles the cookie automatically; we only manage the XSRF header yourself.
- The token-rotation in Step 2 is the classic gotcha — reusing the *Step-1* XSRF token in Step 3 yields `403`s. (Worth a slide: it looks like a permissions problem but is really a stale-token problem.)

In [4]:
# 2.3 OAuth 2.0 stateful: the 3-step flow 

def oauth_stateful_login(oauth_token: str, server_id: int = SERVER_ID):
    """Perform the 3-step OAuth stateful login.
    Returns a requests.Session with the live JSESSIONID cookie,
    plus the current X-XSRF-TOKEN string you must send on each call.
    """
    s = requests.Session()
    s.verify = VERIFY_TLS
    s.headers.update({"Accept": "application/json"})

    # STEP 1: bootstrap — get initial JSESSIONID + XSRF token
    r1 = s.get(f"{BASE}/login/configuration")
    r1.raise_for_status()
    xsrf = r1.headers.get("X-XSRF-TOKEN") or s.cookies.get("XSRF-TOKEN")
    print("Step 1 OK — initial session established.")
    print("   Available servers:", json.dumps(r1.json(), indent=2)[:600])

    # STEP 2: exchange OAuth token for an authenticated session
    r2 = s.get(
        f"{BASE}/accesstoken",
        params={"serverId": server_id},
        headers={
            "Authorization": f"Bearer {oauth_token}",
            "X-XSRF-TOKEN": xsrf,
        },
    )
    r2.raise_for_status()
    # BOTH values rotate here — capture the NEW ones.
    xsrf = r2.headers.get("X-XSRF-TOKEN") or s.cookies.get("XSRF-TOKEN") or xsrf
    print("Step 2 OK — session upgraded to authenticated.")

    return s, xsrf


def catalog_get(session: requests.Session, xsrf: str,
                path: str, params: dict | None = None):
    """STEP 3 helper: authenticated GET using the session + XSRF pair."""
    resp = session.get(
        f"{BASE}{path}",
        params=params or {},
        headers={"X-XSRF-TOKEN": xsrf},
    )
    print(resp.status_code, resp.reason, "-", resp.url)
    return resp

# Usage:
# session, xsrf = oauth_stateful_login(OAUTH_TOKEN)
# resp = catalog_get(session, xsrf, "/public/api/browse/databases",
#                    params={"databaseName": "admin", "serverId": SERVER_ID})
# resp.json()

### 2.4 Local Login (stateful, username/password)

Same session mechanics as 2.3, but the session is established with a **username/password login** instead of an OAuth token exchange. This is the mode the Data Catalog *web UI itself* uses when we type credentials into the login form.

- Works out of the box (no server-side switch), **if** local accounts exist.
- Denodo treats direct password login by external API clients as the legacy path — new automation should prefer the *public* stateless/OAuth modes.
- At an SSO-enforced site like LANL, you may not have a local password at all — in which case this mode simply isn't available to us, and that's fine: 2.3 covers the same need.

**Practical role in our project:** a *fallback for troubleshooting*. If the OAuth flow breaks, a local-login test (by someone who has such an account, e.g. the Denodo admins) isolates whether the problem is OAuth-specific or catalog-wide.

### 2.5 Prerequisite for Everything: `serverId` Discovery

Almost every data endpoint needs `serverId` — the **internal identifier of the Virtual DataPort server registered in the Data Catalog** (a catalog can front several VDP servers).

There are two ways to discover it:

| Endpoint | Auth needed? | Notes |
|---|---|---|
| `/public/api/configuration/servers` | Yes | The official discovery endpoint |
| `/login/configuration` | **No** | Also returns server info — we get it for free in Step 1 of the stateful flow |

If we send the wrong `serverId`, errors can masquerade as auth failures. I think it's worth checking early whether LANL dev and prod use the same internal ID.

In [5]:
# 2.5 Discover registered VDP servers
# Option A (no auth): /login/configuration — see Step 1 output in 2.3.
# Option B (authenticated): the official endpoint.

# resp = catalog_get(session, xsrf, "/public/api/configuration/servers")
# for server in resp.json():
#     print(server)   # note the 'id' field -> use as SERVER_ID

---
## 3. Method 3 - Endpoint Families: What Data Each One Gives Us

Once authenticated (any mode from Section 2), the Swagger page's endpoint groups map to distinct *kinds* of data access. Rough map:

```
Data Catalog REST API
├── 3.1 Browse            -> catalog tree: databases -> views (names, ids)
├── 3.2 View details      -> richest metadata: columns, connectionUris, properties
├── 3.3 Properties        -> Tier 1 / Tier 2 custom metadata (your datacard work)
├── 3.4 Query execution   -> ACTUAL ROWS via VQL through the catalog
├── 3.5 Permissions       -> which views the caller may see
└── 3.6 Export            -> query results as downloadable files
```

### 3.1 Browse Endpoints — Walking the Catalog Tree

`/public/api/browse/...` endpoints let us navigate **database → views** the way the catalog UI's tree does. This is the workhorse behind our snapshot inventory (the 15,014-view dev list / 4,418-view prod list) and the reconciliation notebook.

Typical calls:
- List/inspect databases: `/public/api/browse/databases?databaseName=<db>&serverId=<id>`
- Enumerate views within a database (per the browse group on your Swagger page)

Returns **identifiers and names** — lightweight, ideal for building the full inventory before drilling into details.

In [6]:
# 3.1 Browse the catalog tree 
# resp = catalog_get(session, xsrf, "/public/api/browse/databases",
#                    params={"databaseName": "<your-db>", "serverId": SERVER_ID})
# db_info = resp.json()
# print(json.dumps(db_info, indent=2)[:2000])

# Pattern for a full inventory sweep (pseudo-code — adapt paging params
# to the exact browse endpoints on your Swagger page):
#
# all_views = []
# for db in databases:
#     views = list_views_for(db)        # browse endpoint per database
#     all_views.extend(views)
# print(len(all_views), "views total")  # dev ~15,014 vs prod ~4,418

### 3.2 View-Details Endpoint — The Richest Single Payload

Our investigation already established this is the **highest-value endpoint** on the page. For a single view it returns:

- **Column schema** — field names, types (feeds the OSTI → Genesis/Diana datacard conversion)
- **`connectionUris`** — the actual access URIs into VDP (JDBC/ODBC/REST/OData…) → Section 4
- **Descriptions & properties** — including RICH_TEXT properties, which arrive as **HTML embedded inside JSON strings** (hence your HTML-in-JSON parsing work: treat the value as HTML, strip/parse with e.g. `BeautifulSoup`, don't regex it blind)
- View type, database, and other governance metadata

**Pattern:** browse endpoints give us the *list of view IDs*; view-details gives you *everything about each one*. Inventory first, details second.

In [ ]:
# 3.2 View details 
# VIEW_ID = <id discovered via browse>
# resp = catalog_get(session, xsrf, f"/public/api/views/{VIEW_ID}/details",
#                    params={"serverId": SERVER_ID})
# details = resp.json()
#
# # The three fields your project cares about most:
# schema_cols   = details.get("schema") or details.get("columns")
# conn_uris     = details.get("connectionUris")
# properties    = details.get("properties")
#
# # RICH_TEXT handling — property values may be HTML inside JSON:
# from bs4 import BeautifulSoup
# def rich_text_to_plain(html_string: str) -> str:
#     return BeautifulSoup(html_string or "", "html.parser").get_text(" ", strip=True)

### 3.3 Property-Management Endpoints — Tier 1 / Tier 2 Metadata

These endpoints manage the **custom property groups** attached to catalog elements — the layer where our Tier 1 vs. Tier 2 analysis lives.

Findings from our investigation to keep in mind when presenting:
- The catalog exposed **341 properties** in the snapshot we reconciled.
- **Tier-2 property groups returned empty arrays on all tested views** — i.e., the *structure* exists but is unpopulated (a governance/population gap, not an API failure).
- Reconciliation of downloaded snapshots vs. live API showed **zero real discrepancies** across 15,014 views once NaN-vs-empty-string and list-valued-field bugs were fixed.

**Access pattern:** list property groups → list properties per group → read property values per view (or get them bundled inside view-details, 3.2).

In [7]:
# 3.3 Properties / property groups 
# Adapt paths to the property-management group on your Swagger page:
#
# resp = catalog_get(session, xsrf, "/public/api/property-groups",
#                    params={"serverId": SERVER_ID})
# groups = resp.json()
# for g in groups:
#     print(g.get("id"), g.get("name"))
#
# # Then per group:
# resp = catalog_get(session, xsrf, f"/public/api/property-groups/{GID}/properties",
#                    params={"serverId": SERVER_ID})

### 3.4 Query-Execution Endpoints — Actual Rows, Not Just Metadata

This is the most under-appreciated capability on the Swagger page: **the catalog can execute queries against VDP on your behalf and stream the rows back**. You do not need JDBC/ODBC just to fetch data.

Denodo's own **AI SDK** is the proof-by-existence — it is built entirely on three catalog endpoints:

| Endpoint | Purpose |
|---|---|
| `/public/api/askaquestion/execute` | **Executes VQL queries** and returns result rows |
| `/public/api/askaquestion/data` | Retrieves view **metadata** (schema, descriptions, associations, sample data) |
| `/public/api/views/allowed-identifiers` | Returns the caller's **permitted views** (see 3.5) |

**Implications for our project:**
- A pure-HTTP Python backend (your `denodo.py`) can do *metadata + data* through one API surface and one auth flow.
- Rows come back as JSON — convenient for pandas, no JDBC driver management.
- **Caveats:** results are mediated by the catalog web app (timeouts, response-size limits), so treat this as the path for *interactive/moderate* queries; bulk extraction belongs on the direct-VDP paths in Section 4.

In [ ]:
# 3.4 Execute a VQL query through the catalog 
# VQL = "SELECT * FROM <database>.<view> LIMIT 10"
#
# resp = session.post(
#     f"{BASE}/public/api/askaquestion/execute",
#     headers={"X-XSRF-TOKEN": xsrf, "Content-Type": "application/json"},
#     json={"vql": VQL, "serverId": SERVER_ID},   # confirm exact body schema
# )                                               # in your Swagger page
# rows = resp.json()
#
# import pandas as pd
# df = pd.DataFrame(rows)   # adapt to the actual response envelope
# df.head()

### 3.5 Permissions Endpoint — What the Caller Is Allowed to See

`/public/api/views/allowed-identifiers` returns the set of views the **authenticated user** may access. Two reasons this matters at LANL:

1. Our earlier finding: **permissions are granted per access path, not per dataset** — so the same underlying data may be visible through one path and invisible through another. This endpoint reflects the *catalog-side* resolution of that.
2. **Inventory counts are permission-relative.** If a colleague's sweep returns a different view count than ours, compare `allowed-identifiers` before assuming a dev/prod or snapshot discrepancy.

In [ ]:
# 3.5 Allowed identifiers 
# resp = catalog_get(session, xsrf, "/public/api/views/allowed-identifiers",
#                    params={"serverId": SERVER_ID})
# allowed = resp.json()
# print(len(allowed), "views visible to this account")

### 3.6 Export — Query Results as Files

The catalog UI's query screen has **Execute** and **Export** buttons — Export saves the query results in our chosen format (e.g., CSV/Excel). The corresponding endpoints appear in the API surface, so the same "run query → download file" flow can be scripted.

Where it fits: **one-off deliverables** for stakeholders ("here's the CSV of X") without standing up a JDBC pipeline. For repeated/large extractions, prefer direct VDP access (Section 4).

---


---
## 4. Method 4 — Indirect Access: `connectionUris` → Direct VDP Access

The `connectionUris` field in view-details is **not data — it's a signpost**. It tells us where to query **Virtual DataPort directly**, bypassing the catalog web app for the actual data movement:

| Channel | What it looks like | Typical client | Best for |
|---|---|---|---|
| **JDBC** | `jdbc:denodo://<vdp-host>:9999/<database>` | Python (`jaydebeapi`/JayDeBeApi + Denodo driver), Java, BI tools | Bulk pulls, pandas pipelines |
| **ODBC** | DSN against VDP's ODBC/PostgreSQL-compatible port | Excel, Tableau, Power BI, `pyodbc` | Analyst tooling |
| **REST web services** | Per-view resources published from VDP | Any HTTP client | App integration; supports Execute/Export per resource |
| **OData** | OData 4 service over views | Power BI, Excel, generic OData clients | Standards-based consumption |
| **GraphQL** | GraphQL service over views | Modern web front-ends | Selective-field fetching |




---
## 5. Summary 

### Which access method for which job?

| The task | Use |
|---|---|
| Explore an unfamiliar endpoint, inspect response shapes | **1. Swagger UI "Try it out"** |
| Automated metadata harvesting (inventories, reconciliation, datacards) | **2.3 OAuth stateful** session + **3.1/3.2/3.3** endpoints |
| Service-to-service automation with no cookie bookkeeping | **2.2 OAuth stateless** *(if enabled server-side — confirm with Maxen)* |
| Fetch actual rows for analysis in pandas (moderate size) | **3.4** `/public/api/askaquestion/execute` |
| Respect/verify visibility rules | **3.5** `allowed-identifiers` |
| One-off file deliverable of query results | **3.6 Export** |
| Bulk/repeated data extraction, BI tools | **4.** direct VDP via JDBC/ODBC/OData from `connectionUris` |


